# Preprocessing: from a question to a checked Cypher query

In this project "preprocessing" is everything that is computed **before a language model is called**, plus the
deterministic check of what the model returns. It follows the two papers the team chose in Milestone 1:

- **T2CSS** (Wan et al., *Decision Support Systems* 199, 2025): build the prompt from the question, the relevant part
  of a semantic schema and the relevant Cypher rules.
- **CYGNET** (Tomczak, arXiv 2606.04645, 2026): pass a generated query through deterministic gates before it runs.

The notebook walks **one question through all five steps** and then measures how the filter behaves on the five
questions we have. Preprocessing is identical for every question and every model, so one worked example is enough;
the code lives in `src/t2css.py`, `src/validate.py` and `src/build_schema.py`.

| Step | What happens | Function |
|---|---|---|
| 0 | export the schema of the graph once | `build_schema.build()` → `data/schema.json` |
| 1 | question → vector | `t2css.question_vector` |
| 2 | pick the schema triples closest to the question | `t2css.rank_triples`, `select_triples` |
| 3 | pick the Cypher rules closest to the question | `t2css.rank_rules`, `select_rules` |
| 4 | assemble the prompt | `t2css.build_system_prompt` |
| 5 | check the generated query, then run it | `validate.validate`, `validate_and_run` |

In [1]:
import ast, json, os, re, sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore", message=".*IProgress not found.*")

root = Path.cwd()
while not (root / "pyproject.toml").exists():
    root = root.parent
os.chdir(root)
sys.path.insert(0, "src")

import numpy as np
import pandas as pd
import t2css, validate
from prompt import SYSTEM_PROMPT

pd.set_option("display.max_colwidth", 110)
questions = [json.loads(l) for l in open("data/questions.jsonl", encoding="utf-8-sig") if l.strip()]
Q = next(q for q in questions if q["id"] == "q04")
print(Q["id"], "-", Q["question"])

q04 - Which 5 compounds bind the most genes associated with osteoporosis but are NOT known to treat it? Return name and shared gene count; break ties alphabetically by name.


## Step 0. The semantic schema

T2CSS does not give the model the graph itself but a *semantic schema*: classes, properties and relations, without
instance data. For Hetionet this is small. `src/build_schema.py` reads it from the live server and stores it as
`data/schema.json`: 11 node labels, 24 relationship types written as triples `(source, TYPE, target)`, each with a
one-line plain-English gloss. The gloss is the text that gets embedded in step 2.

In [2]:
schema = t2css.load_schema()
print(schema["server"], "| exported", schema["exported"])
print(len(schema["labels"]), "labels |", len(schema["triples"]), "triples |", len(schema["property_keys"]), "property keys")
pd.DataFrame(schema["triples"])[["source", "type", "target", "gloss"]]

{'name': 'Neo4j Kernel', 'version': '3.5.12', 'edition': 'community'} | exported 2026-11-20
11 labels | 24 triples | 23 property keys


,source,type,target,gloss
0,Disease,ASSOCIATES_DaG,Gene,disease is associated with gene
1,Compound,BINDS_CbG,Gene,compound (drug) binds gene (its protein target)
2,Compound,CAUSES_CcSE,SideEffect,compound (drug) causes side effect
3,Gene,COVARIES_GcG,Gene,gene covaries with gene (evolutionary rate covariation)
4,Anatomy,DOWNREGULATES_AdG,Gene,anatomy (tissue) downregulates gene expression
5,Compound,DOWNREGULATES_CdG,Gene,compound (drug) downregulates gene expression
6,Disease,DOWNREGULATES_DdG,Gene,disease downregulates gene expression
7,Anatomy,EXPRESSES_AeG,Gene,anatomy (tissue) expresses gene
8,PharmacologicClass,INCLUDES_PCiC,Compound,pharmacologic class includes compound (drug)
9,Gene,INTERACTS_GiG,Gene,gene interacts with gene (protein-protein interaction)


## Step 1. Question → vector

Paper, eq. (3): after tokenisation, stop-word removal and stemming, the question vector is the **mean of the vectors
of its words**. The paper does not name the word vectors it uses.

Here: lower-casing and stop-word removal, then `potion-base-8M` through the `model2vec` library. It is a *static*
embedding model, so a sentence vector is exactly a mean of token vectors, it runs on CPU, and the same text always
gives the same vector. Stemming is skipped because the model works on sub-word tokens.

In [3]:
print("tokens:", t2css.tokens(Q["question"]))
v = t2css.question_vector(Q["question"])
print("vector:", v.shape, v.dtype, "| length", round(float(np.linalg.norm(v)), 4), "| first values", np.round(v[:5], 4))
print("same vector on a second call:", bool(np.array_equal(v, t2css.question_vector(Q["question"]))))

tokens: ['5', 'compounds', 'bind', 'most', 'genes', 'associated', 'osteoporosis', 'not', 'known', 'treat', 'name', 'shared', 'gene', 'count', 'break', 'ties', 'alphabetically', 'name']


vector: (256,) float32 | length 1.0 | first values [ 0.0896 -0.0576 -0.2932 -0.2409  0.1138]
same vector on a second call: True


## Step 2. Schema mapping

Paper, eq. (4): cosine similarity between the question vector and the schema triples; the context given to the model
is the matched concept "and its three-order concepts". No threshold and no number of triples is stated.

Below: all 24 triples ranked for the example question. `needed` marks the relationship types that the reference
query for this question actually uses.

In [4]:
def gold_types(q):
    return sorted(set(re.findall(r"\[\w*:(\w+)", q["gold_cypher"])))

ranked = pd.DataFrame(t2css.rank_triples(Q["question"], schema))
ranked["needed"] = ranked["type"].isin(gold_types(Q)).map({True: "yes", False: ""})
print("reference query:", Q["gold_cypher"])
ranked[["rank", "score", "source", "type", "target", "needed"]].head(15)

reference query: MATCH (d:Disease {name:'osteoporosis'}) MATCH (c:Compound)-[:BINDS_CbG]->(g:Gene)<-[:ASSOCIATES_DaG]-(d) WHERE NOT (c)-[:TREATS_CtD]->(d) RETURN c.name AS name, count(DISTINCT g) AS shared_genes ORDER BY shared_genes DESC, name ASC LIMIT 5


,rank,score,source,type,target,needed
0,1,0.5210,Disease,ASSOCIATES_DaG,Gene,yes
1,2,0.5060,Compound,BINDS_CbG,Gene,yes
2,3,0.4640,Gene,INTERACTS_GiG,Gene,
3,4,0.4513,Compound,UPREGULATES_CuG,Gene,
4,5,0.4453,Gene,REGULATES_GrG,Gene,
5,6,0.4366,Compound,DOWNREGULATES_CdG,Gene,
6,7,0.4340,Gene,PARTICIPATES_GpMF,MolecularFunction,
7,8,0.4248,Disease,UPREGULATES_DuG,Gene,
8,9,0.4029,Disease,DOWNREGULATES_DdG,Gene,
9,10,0.3945,Gene,COVARIES_GcG,Gene,


### The "three-order" neighbourhood on Hetionet

The paper adds the concepts up to three steps away from the matched ones. The cell below does exactly that with the
`order` parameter: start from the top-*k* triples and add every triple inside the *order*-step neighbourhood of their
labels.

In [5]:
labels = sorted(schema["labels"])
dist = {a: {b: next(o for o in range(12) if b in t2css.neighbourhood({a}, schema, o)) for b in labels} for a in labels}
print("largest distance between two node labels in the schema graph:", max(max(d.values()) for d in dist.values()))
rows = [{"k": k, **{f"order={o}": len(t2css.select_triples(Q["question"], schema, k=k, order=o)) for o in (0, 1, 2, 3)}}
        for k in (1, 3, 6)]
print("number of triples selected for the example question, out of", len(schema["triples"]))
pd.DataFrame(rows).set_index("k")

largest distance between two node labels in the schema graph: 3
number of triples selected for the example question, out of 24


,order=0,order=1,order=2,order=3
k,,,,
1,1,22,24,24
3,3,24,24,24
6,6,24,24,24


## Step 3. Cypher syntax selection

Paper, eq. (5): the rules are chosen by a conditional probability `Z(q | T')` computed with Bayes' rule; how the
probabilities are estimated is not described. Estimating them needs many question-query pairs (the paper's own
fine-tuned baseline was trained on 11,042 pairs). We have five questions, so nothing can be estimated.

Here: a hand-written rule base for Neo4j 3.5 (`data/cypher_rules.json`). Two rules are always included; the others are
ranked by the same cosine similarity, between the question and a one-line description of the rule.

In [6]:
rules = t2css.load_rules()
print(len(rules), "rules | always on:", [r["id"] for r in rules if r["always"]])
pd.DataFrame(t2css.rank_rules(Q["question"], rules))[["rank", "score", "id", "text"]]

11 rules | always on: ['dialect', 'direction']


,rank,score,id,text
0,1,0.5129,lookup,"Look an entity up by exact name with a property map, e.g. (d:Disease {name: '...'}); do not use CONTAINS o..."
1,2,0.4033,multi_hop,"For multi-step questions chain patterns through the shared node, e.g. (a)-[:R1]->(shared)<-[:R2]-(b), inst..."
2,3,0.2874,count,"Count with count(DISTINCT x) and give the result an alias, e.g. count(DISTINCT g) AS shared_genes."
3,4,0.2745,one_hop,For a direct relation use one MATCH pattern with one relationship type and RETURN DISTINCT the name of the...
4,5,0.2657,case,Names are case-sensitive; if the spelling may differ compare with toLower(n.name) = toLower('...').
5,6,0.1900,aggregate_filter,"To filter on an aggregate, compute it in WITH and filter after it: WITH a, count(DISTINCT b) AS n WHERE n ..."
6,7,0.1733,projection,"Return only the columns the question asks for, each with an explicit alias (AS name)."
7,8,0.1517,ranking,"For top-N questions use ORDER BY <count> DESC, then name ASC as a tie-break, then LIMIT N with exactly the..."
8,9,0.1115,negation,"To exclude entities that already have a relationship use a pattern predicate: WHERE NOT (a)-[:TYPE]->(b), ..."


## Step 4. Prompt assembly

Paper, eq. (1): `P = question + C(O) + selected rules`. We keep the opening and closing lines of the team's shared
system prompt (`src/prompt.py`) and replace only its schema description, so the variants differ in nothing else.

`src/t2css.py` writes five variants per question to `data/t2css_context.jsonl`. The first four are the input
strategies of the paper's own ablation (section 6.4.2); the fifth switches the similarity filter on.

In [7]:
variants = {name: t2css.build_system_prompt(Q["question"], schema, rules, **opts) for name, opts in t2css.VARIANTS.items()}
print(pd.Series({"team baseline (src/prompt.py)": len(SYSTEM_PROMPT.split()),
                 **{k: len(v.split()) for k, v in variants.items()}}, name="words").to_string())
print("\n----- s4_filtered (k=6 triples, 3 optional rules) -----\n")
print(variants["s4_filtered"])

team baseline (src/prompt.py)    168
s1_question_only                  94
s2_rules                         311
s3_schema                        295
s4_schema_rules                  512
s4_filtered                      255

----- s4_filtered (k=6 triples, 3 optional rules) -----

You answer questions about Hetionet, a biomedical knowledge graph stored in Neo4j 3.5.
Use the run_cypher tool to query it. Base every answer on query results, never on memory.

Cypher dialect: Neo4j 3.5. Do not use newer syntax such as EXISTS { ... } subqueries.
Queries are read-only. Results are capped at 50 rows; check the 'truncated' flag.

Node labels (all have name and identifier): Compound, Disease, Gene.
Disease names are lowercase (e.g. 'bile duct cancer'); compound names are capitalised.
Relationship types (direction matters):
- (:Disease)-[:ASSOCIATES_DaG]->(:Gene)  disease is associated with gene
- (:Compound)-[:BINDS_CbG]->(:Gene)  compound (drug) binds gene (its protein target)
- (:Compound)-[:DOW

## Step 5. Check the query, then run it

T2CSS validates the generated query "against the semantic schema" before execution; CYGNET makes this concrete as a
chain of deterministic gates. We keep three, cheapest first:

1. **vocabulary**: every label, relationship type and property must exist in `data/schema.json` (no network);
2. **direction**: a relationship must be written in the direction stored in the schema (no network);
3. **explain**: the server parses and plans the query with `EXPLAIN`; nothing is executed.

A query is run only if all three pass. First the reference query of the example question:

In [8]:
report = validate.validate_and_run(Q["gold_cypher"], schema)
print("ok:", report["ok"], "| gates:", report["gates"], "| planner estimate:", round(report["estimated_rows"], 2), "rows")
got = report["result"]["rows"]
print("result equals the stored reference result:", got == Q["gold_result"])
pd.DataFrame(got)

ok: True | gates: {'vocabulary': [], 'direction': [], 'explain': []} | planner estimate: 3.57 rows
result equals the stored reference result: True


,name,shared_genes
0,Danazol,3
1,Diethylstilbestrol,3
2,Estriol,3
3,Estrone,3
4,Levonorgestrel,3


Deliberately broken versions of a simple query show which gate reports what. All gates are run here (no
short-circuit) to make the comparison visible. Note the fourth row: a reversed relationship is valid Cypher, the
server plans it without complaint and it would simply return nothing; only the direction gate sees it.

In [9]:
broken = {
    "unknown label": "MATCH (c:Drug)-[:TREATS_CtD]->(d:Disease {name:'asthma'}) RETURN c.name",
    "unknown relationship type": "MATCH (c:Compound)-[:CURES]->(d:Disease {name:'asthma'}) RETURN c.name",
    "unknown property": "MATCH (c:Compound)-[:TREATS_CtD]->(d:Disease {name:'asthma'}) RETURN c.brand_name",
    "reversed direction": "MATCH (d:Disease {name:'asthma'})-[:TREATS_CtD]->(c:Compound) RETURN c.name",
    "syntax error": "MATCH (c:Compound)-[:TREATS_CtD]->(d:Disease {name:'asthma'} RETURN c.name",
}
rows = []
for name, query in broken.items():
    r = validate.validate(query, schema, short_circuit=False)
    rows.append({"mistake": name, **{g: "; ".join(p)[:95] or "-" for g, p in r["gates"].items()}})
pd.DataFrame(rows).set_index("mistake")

,vocabulary,direction,explain
mistake,,,
unknown label,unknown label :Drug,"TREATS_CtD goes (Compound)->(Disease), but the query writes (Drug)->(Disease)",The provided label is not in the database. [UnknownLabelWarning]
unknown relationship type,unknown relationship type :CURES,-,The provided relationship type is not in the database. [UnknownRelationshipTypeWarning]
unknown property,label Compound has no property 'brand_name' (used as c.brand_name),-,The provided property key is not in the database [UnknownPropertyKeyWarning]
reversed direction,-,"TREATS_CtD goes (Compound)->(Disease), but the query writes (Disease)->(Compound)",-
syntax error,-,-,"Invalid input 'R': expected whitespace, comment or ')' (line 1, column 70 (offset: 69))"


The same gates on **real model output**: every query the five models sent during the pilot run
(`results/pilot_20261025_131041.jsonl`), compared with what the server actually answered at the time.

In [10]:
pilot = [json.loads(l) for l in open("results/pilot_20261025_131041.jsonl", encoding="utf-8-sig") if l.strip()]
rows = []
for rec in pilot:
    steps = rec.get("steps") or []
    steps = ast.literal_eval(steps) if isinstance(steps, str) else steps
    for i, s in enumerate(steps):
        r = validate.validate(s["cypher"], schema, short_circuit=False)
        rows.append({"model": rec["model"], "question": rec["qid"], "step": i,
                     "server error in pilot": bool((s.get("result") or {}).get("error")),
                     **{g: bool(p) for g, p in r["gates"].items()},
                     "report": "; ".join(p for ps in r["gates"].values() for p in ps)[:110]})
checked = pd.DataFrame(rows)
print(len(checked), "queries |", int(checked["server error in pilot"].sum()), "failed on the server in the pilot |",
      "flagged by: vocabulary", int(checked["vocabulary"].sum()), ", direction", int(checked["direction"].sum()),
      ", explain", int(checked["explain"].sum()))
print("queries that passed every gate but failed on the server:",
      int((checked["server error in pilot"] & ~checked[["vocabulary", "direction", "explain"]].any(axis=1)).sum()))
checked[checked[["vocabulary", "direction", "explain"]].any(axis=1)].drop(columns=["vocabulary", "direction", "explain"])

40 queries | 2 failed on the server in the pilot | flagged by: vocabulary 1 , direction 0 , explain 3
queries that passed every gate but failed on the server: 0


,model,question,step,server error in pilot,report
5,qwen3.8-27b,q02,0,False,unknown property key 'score' (used as r.score); The provided property key is not in the database [UnknownP...
17,mistral-small-24b,q04,0,True,"Invalid input 'U': expected 'n/N' (line 1, column 173 (offset: 172))"
24,gpt-oss-120b,q04,1,True,"Variable `d` not defined (line 4, column 52 (offset: 200))"


## How the filter behaves on our five questions

For each question: where the relationship types and the rules that its reference query needs land in the ranking.
A filter that keeps the top *k* loses everything ranked below *k*. The rules a reference query needs are read off the
query text (`ORDER BY` + `LIMIT` → `ranking`, `NOT (` → `negation`, and so on).

In [11]:
def needed_rules(q):
    c = q["gold_cypher"]
    need = []
    if "{name:" in c.replace(" ", ""): need.append("lookup")
    if len(re.findall(r"\[\w*:\w+", c)) >= 2: need.append("multi_hop")
    if "count(" in c.lower(): need.append("count")
    if "ORDER BY" in c and "LIMIT" in c: need.append("ranking")
    if "NOT (" in c: need.append("negation")
    return need

rows = []
for q in questions:
    tpos = {t["type"]: t["rank"] for t in t2css.rank_triples(q["question"], schema)}
    rpos = {r["id"]: r["rank"] for r in t2css.rank_rules(q["question"], rules)}
    tr, ru = {t: tpos[t] for t in gold_types(q)}, {r: rpos[r] for r in needed_rules(q)}
    rows.append({"question": q["id"], "needed triples: rank of 24": tr, "smallest k": max(tr.values(), default=0),
                 "needed rules: rank of 9": ru, "smallest rule_k": max(ru.values(), default=0)})
cover = pd.DataFrame(rows).set_index("question")
print("questions fully covered by the filter:",
      {f"k={k}": int((cover["smallest k"] <= k).sum()) for k in (3, 6, 10, 14, 24)}, "of", len(cover))
cover

questions fully covered by the filter: {'k=3': 4, 'k=6': 4, 'k=10': 4, 'k=14': 5, 'k=24': 5} of 5


,needed triples: rank of 24,smallest k,needed rules: rank of 9,smallest rule_k
question,,,,
q01,{'TREATS_CtD': 1},1,{'lookup': 1},1
q02,{'TREATS_CtD': 1},1,{'lookup': 1},1
q03,"{'ASSOCIATES_DaG': 1, 'BINDS_CbG': 3}",3,"{'lookup': 1, 'multi_hop': 2, 'count': 3, 'ranking': 8}",8
q04,"{'ASSOCIATES_DaG': 1, 'BINDS_CbG': 2, 'TREATS_CtD': 14}",14,"{'lookup': 1, 'multi_hop': 2, 'count': 3, 'ranking': 8, 'negation': 9}",9
q05,{},0,{'lookup': 1},1


## What this shows, and what we do with it

**Findings on this graph** (numbers are in the cells above):

- The schema is 24 triples. Our baseline prompt already contains all of it in about 170 words, so the problem the
  filter was designed for (a schema that does not fit the context window) does not occur here.
- The paper's three-order neighbourhood selects the **whole schema**: the schema graph has diameter 3, and for the
  example question one step from its top three triples already selects all 24.
- A top-*k* filter does cut the prompt, but for the question with a negation it ranks the negated relation
  (`TREATS_CtD`) far down, so *k* has to be 14 of 24 before every reference query is covered.
- Rule selection by similarity ranks `ranking` and `negation` near the bottom for exactly the questions that need them.
- The gates behave as intended on real model output: both queries that failed on the server in the pilot are caught
  by `EXPLAIN`, and the vocabulary gate finds one invented property (`r.score`) that the server had executed silently.
  The other 37 queries pass every gate.

**Decisions.** Both methods stay in the pipeline. The default T2CSS prompt for the experiments is `s4_schema_rules`
(full schema and all rules in the T2CSS layout). `s4_filtered` is kept as a separate arm, and `s1`-`s3` reproduce the
paper's ablation, so Milestones 4 and 5 can measure the effect of each component instead of assuming it. All variants
are pre-computed per question, so every model receives byte-identical text.

**Where we deviate from the papers**

| Step | Paper | Here | Reason |
|---|---|---|---|
| Question vector | tokenise, remove stop words, stem; mean of word vectors; model not named | remove stop words; mean of static token vectors (`potion-base-8M`) | same construction; sub-word tokens replace stemming; CPU-only and deterministic |
| Schema mapping | cosine similarity; matched concept plus three-order concepts; no threshold given | cosine similarity to a gloss per relationship type; top-*k*; `order` available as a parameter | three-order = whole schema on Hetionet |
| Syntax selection | conditional probability by Bayes' rule; estimation not described | similarity to a one-line rule description; two always-on rules | five questions are not enough to estimate probabilities |
| Validation (T2CSS) | validate against the schema, let the LLM refine; number of rounds not stated | three deterministic gates; the repair loop is part of the agent run (Milestone 4) | gates need no model, so they belong to preprocessing |
| Gates (CYGNET) | regex, ANTLR parser, `EXPLAIN`, mirror graph, cost gate; up to 3 repair attempts | vocabulary, direction, `EXPLAIN`; planner row estimate recorded, no threshold | the ANTLR grammar rejected valid Neo4j 3.5 syntax (`exists(n.prop)`) in our test of `antlr4-cypher` 0.1.2; a mirror graph needs a writable database and the public server is read-only; `EXPLAIN` does not see a reversed relationship, hence our direction gate |

We therefore describe the implementation as *inspired by* T2CSS and CYGNET, not as a replication.

In [12]:
ctx = [json.loads(l) for l in open("data/t2css_context.jsonl", encoding="utf-8")]
print("data/t2css_context.jsonl:", len(ctx), "questions | fields:", list(ctx[0]))
print("prompt variants:", list(ctx[0]["system_prompts"]))

data/t2css_context.jsonl: 5 questions | fields: ['id', 'question', 'tokens', 'embedding_model', 'k', 'rule_k', 'triple_ranking', 'rule_ranking', 'system_prompts']
prompt variants: ['s1_question_only', 's2_rules', 's3_schema', 's4_schema_rules', 's4_filtered']
